# ComputerUseAgent — Browser Automation (v1.0.8)

Drive a real browser by showing screenshots to a vision-capable LLM and
parsing structured action commands back. Anthropic's `computer-use` API
works natively; OpenAI, Bedrock, Gemini all work via plain-text fallback.

Use cases:
- **Web apps with no API** — booking flights, internal SaaS, legacy dashboards
- **Form filling** at scale, with human review
- **End-to-end testing** that adapts when the UI shifts
- **Data extraction** from sites without a public API

This notebook shows four real-life patterns. We use a `MockBrowserSession`
throughout so the notebook runs offline; swap in `PlaywrightBrowserSession.launch()`
for production.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent.computer_use import (
    ComputerUseAgent, MockBrowserSession, parse_action,
)

## Example 1 — Look up a product price

Goal: *navigate to apple.com, find the iPhone 15 Pro page, extract the starting price.*

We script the LLM's responses so the notebook is reproducible. In production,
the model would actually look at each screenshot.

In [2]:

class ScriptedLLM:
    def __init__(self, replies):
        self.replies = replies
        self._i = 0
    def complete(self, *, messages, **_):
        text = self.replies[min(self._i, len(self.replies) - 1)]
        self._i += 1
        return text

llm = ScriptedLLM([
    "I'll start by navigating to apple.com.\nACTION: navigate https://apple.com",
    "I see the homepage. iPhone is in the top nav.\nACTION: click 240,80",
    "On the iPhone hub. Click into iPhone 15 Pro.\nACTION: click 600,400",
    "Found the page. Starting price is $999 (visible in the buy section).\nACTION: done iPhone 15 Pro starts at $999.",
])

browser = MockBrowserSession()
agent = ComputerUseAgent(
    llm=llm,
    browser=browser,
    goal="Find the starting price of the iPhone 15 Pro on apple.com",
    max_iterations=8,
)
result = agent.run()

print(f'status: {result.status}')
print(f'final answer: {result.final_text}')
print(f'iterations: {result.iterations}')
print()
print('action history:')
for r in result.action_history:
    print(f'  {r.iteration}: {r.action.kind.value:<10} {r.action.args}')

status: done
final answer: iPhone 15 Pro starts at $999.
iterations: 4

action history:
  0: navigate   {'url': 'https://apple.com'}
  1: click      {'x': 240, 'y': 80}
  2: click      {'x': 600, 'y': 400}
  3: done       {'final_text': 'iPhone 15 Pro starts at $999.'}


## Example 2 — Form filling with intermediate results

Goal: *fill out a contact form, submit it, capture the confirmation.*

Note how the agent uses `key Enter` after typing, and how a `done` action
captures the page's confirmation message.

In [3]:
form_browser = MockBrowserSession()
form_llm = ScriptedLLM([
    'Click the Name field.\nACTION: click 400,200',
    'ACTION: type "Rahul Raj"',
    'ACTION: key Tab',
    'ACTION: type "rahul@example.com"',
    'ACTION: key Tab',
    'ACTION: type "Hello, I would like to discuss our partnership."',
    'ACTION: click 400,500',
    'I see a confirmation: "Thanks! We\'ll get back to you in 1 business day."\nACTION: done Form submitted; confirmation: We will get back in 1 business day.',
])

form_agent = ComputerUseAgent(
    llm=form_llm,
    browser=form_browser,
    goal="Fill out the contact form with name 'Rahul Raj', email 'rahul@example.com', and the message 'Hello, I would like to discuss our partnership.'",
    max_iterations=15,
)
form_result = form_agent.run()
print(form_result.final_text)
print()
print(f'browser ran {len(form_browser.calls)} commands:')
for c in form_browser.calls:
    print(f'  {c[0]:<10} {c[1]}')

Form submitted; confirmation: We will get back in 1 business day.

browser ran 15 commands:
  screenshot {}
  click      {'x': 400, 'y': 200}
  screenshot {}
  type       {'text': 'Rahul Raj'}
  screenshot {}
  key        {'key': 'Tab'}
  screenshot {}
  type       {'text': 'rahul@example.com'}
  screenshot {}
  key        {'key': 'Tab'}
  screenshot {}
  type       {'text': 'Hello, I would like to discuss our partnership.'}
  screenshot {}
  click      {'x': 400, 'y': 500}
  screenshot {}


## Example 3 — Recovering from a failed action

Goal: *click a button that's actually offscreen.*

We make `click` raise on the first attempt; the agent sees the error
in its history and adapts. This is the recovery pattern that makes
computer-use agents production-ready.

In [4]:
class FlakyBrowser(MockBrowserSession):
    def __init__(self):
        super().__init__()
        self._click_count = 0
    def click(self, x, y):
        self._click_count += 1
        if self._click_count == 1:
            raise RuntimeError('Element not visible at (100, 100)')
        return super().click(x, y)

recover_llm = ScriptedLLM([
    'ACTION: click 100,100',  # will fail
    'ACTION: scroll 0 400',   # scroll down to bring element into view
    'ACTION: click 100,100',  # try again
    'ACTION: done Successfully clicked after scrolling.',
])

recover_browser = FlakyBrowser()
recover_agent = ComputerUseAgent(
    llm=recover_llm, browser=recover_browser, goal='click submit', max_iterations=8,
)
recover_result = recover_agent.run()

print(f'status: {recover_result.status}')
print(f'final: {recover_result.final_text}')
print()
print('action history with errors:')
for r in recover_result.action_history:
    err = f' ✗ {r.error}' if r.error else ' ✓'
    print(f'  {r.iteration}: {r.action.kind.value} {r.action.args}{err}')

status: done
final: Successfully clicked after scrolling.

action history with errors:
  0: click {'x': 100, 'y': 100} ✗ Element not visible at (100, 100)
  1: scroll {'dx': 0, 'dy': 400} ✓
  2: click {'x': 100, 'y': 100} ✓
  3: done {'final_text': 'Successfully clicked after scrolling.'} ✓


## Example 4 — Anthropic native computer-use shape

When using Claude with the native `computer-use` tool, responses come back
as structured `tool_use` blocks instead of plain text. The parser handles
both shapes; you don't change anything in your agent code.

In [5]:
anthropic_block = {
    'type': 'tool_use',
    'name': 'computer',
    'input': {
        'action': 'left_click',
        'coordinate': [320, 180],
        'rationale': 'Clicking the search icon in the top-right.',
    },
}
action = parse_action(anthropic_block)
print(f'kind: {action.kind.value}')
print(f'args: {action.args}')
print(f'rationale: {action.rationale}')

kind: click
args: {'x': 320, 'y': 180}
rationale: Clicking the search icon in the top-right.


## Production setup

Swap `MockBrowserSession` for the real Playwright session:

```python
# pip install playwright
# playwright install chromium

from shipit_agent.computer_use import (
    ComputerUseAgent, PlaywrightBrowserSession,
)

with PlaywrightBrowserSession.launch(
    headless=True,
    viewport_size=(1280, 720),
) as browser:
    agent = ComputerUseAgent(
        llm=opus_llm,
        browser=browser,
        goal='Find the cheapest direct flight from SFO to JFK on May 20',
        max_iterations=15,
    )
    result = agent.run()
    print(result.final_text)
```

The `with` block guarantees the browser closes on exit, even if the agent
raises. Headless `True` is the default; pass `False` to watch it run.

## Combine with other v1.0.8 features

**Pair with the Verifier network** to gate destructive actions (e.g. don't
click 'Delete account'):

```python
from shipit_agent.verifier import VerifierNetwork

verifier = VerifierNetwork(llm=haiku_llm, goal='Look up product prices')
# Wrap any tool — the ComputerUseAgent's actions go through the browser,
# but if you also expose a regular `click_button(name)` tool, the verifier
# can veto destructive ones at the Agent level.
```

**Pair with Time-Travel Replay** to debug failed runs — every action and
screenshot is in `result.action_history`, so you can fork from any
iteration and try a different prompt.